
# Building a Network in PyTorch

Scikit-learn is fantastic for standard algorithms, but for Deep Learning, we need more power. We need libraries that can:

1.  **Run on GPUs** (Deep learning relies on large matrix operations, where GPUs are typically much faster).
2.  **Calculate Gradients Automatically** (Backpropagation is hard to code by hand).

The two industry giants are **PyTorch** (Meta) and **TensorFlow** (Google). We will focus on **PyTorch**, as it is the standard for research and education due to its readable, "Pythonic" style.

## Tensors and Operations

The fundamental building block of PyTorch is the **Tensor**. Think of a Tensor as a NumPy array that can live on a GPU and remembers how it was created.

In [ ]:
import torch
import numpy as np

# 1. Creating Tensors
x_np = np.array([1, 2, 3])
x_pt = torch.tensor([1, 2, 3])  # From list
x_ones = torch.ones(2, 3)       # Shape (2, 3)

print(f"PyTorch Tensor: {x_pt}")

# 2. Operations (Similar to NumPy)
y_pt = x_pt * 2 + 5
print(f"Math Result: {y_pt}")

# 3. GPU Support (Check if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

# Move tensor to device
x_pt = x_pt.to(device)

## Defining the Architecture (`nn.Module`)

In PyTorch, a neural network is a Python class that inherits from `nn.Module`. You must define two things:

1.  `__init__`: Define the layers (e.g., Linear, ReLU).
2.  `forward`: Define how data moves through those layers.

In [ ]:
import torch.nn as nn

class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        # Define layers
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid() # For binary classification

    def forward(self, x):
        # Define the flow
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        x = self.sigmoid(x)
        return x

# Initialize
model = SimpleMLP(input_dim=2, hidden_dim=10, output_dim=1)
print(model)

## The Training Loop (The "Boilerplate")

Unlike Scikit-learn's `model.fit()`, PyTorch requires you to write the training loop manually. This gives you total control.

The standard loop has **5 Steps**:

1.  **Zero Gradients**: Reset gradients from the previous step.
2.  **Forward Pass**: Compute prediction.
3.  **Compute Loss**: How bad is the prediction?
4.  **Backward Pass**: Calculate gradients (Backpropagation).
5.  **Optimizer Step**: Update weights.

## Practical Demonstration: PyTorch Classification

We will solve the "Moons" dataset using our custom PyTorch model.

### Prepare Data (Tensors)

PyTorch models expect Float32 Tensors, not NumPy arrays.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Generate Data
X_np, y_np = make_moons(n_samples=500, noise=0.2, random_state=42)

# Split first, then scale using training data only
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_test_np = scaler.transform(X_test_np)

# Convert to Tensors (Float32)
# Note: unsqueeze(1) changes shape from (N,) to (N, 1) to match output layer
X_train = torch.tensor(X_train_np, dtype=torch.float32)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test_np, dtype=torch.float32).unsqueeze(1)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

### Setup Training

In [ ]:
import torch.optim as optim

# 1. Model
model = SimpleMLP(input_dim=2, hidden_dim=16, output_dim=1)

# 2. Loss Function (Binary Cross Entropy)
criterion = nn.BCELoss()

# 3. Optimizer (Adam)
optimizer = optim.Adam(model.parameters(), lr=0.01)

### The Loop

In [ ]:
epochs = 1000
losses = []

for epoch in range(epochs):
    # --- Step 1: Zero Gradients ---
    optimizer.zero_grad()

    # --- Step 2: Forward Pass ---
    y_pred = model(X_train)

    # --- Step 3: Compute Loss ---
    loss = criterion(y_pred, y_train)
    losses.append(loss.item())

    # --- Step 4: Backward Pass ---
    loss.backward() # Calculates dLoss/dWeights

    # --- Step 5: Update Weights ---
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# Plot Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epochs")
plt.show()

### Evaluation (No Gradients)

When evaluating, switch to `model.eval()` and use `torch.no_grad()` to save memory.

In [ ]:
model.eval()
with torch.no_grad():
    # Forward pass on test data
    test_preds = model(X_test)
    # Convert probabilities to 0 or 1
    predicted_classes = (test_preds > 0.5).float()
    # Calculate accuracy
    accuracy = (predicted_classes == y_test).float().mean()

print(f"Test Accuracy: {accuracy:.4f}")

## Exercises

### Changing the Architecture

Modify the `SimpleMLP` class to be **Deeper**.

-   Add a second hidden layer (`layer3`).
-   Use =ReLU activation between them.
-   Train the new model on the same data.

### Regression with PyTorch

PyTorch isn't just for classification.

-   Generate regression data (`y = 3x + 2`).
-   Create a model **without** Sigmoid at the end (Linear output).
-   Use `nn.MSELoss()` instead of `BCELoss`.

## Summary

1.  **Tensors**: The math engine. Always check shapes and types (`float32`).
2.  **nn.Module**: The class where you define layers (`__init__`) and flow (`forward`).
3.  **The Loop**: Zero Grad $\rightarrow$ Forward $\rightarrow$ Loss $\rightarrow$ Backward $\rightarrow$ Step.